Downloaded data from https://www.kaggle.com/datasets/wyattowalsh/basketball

Step 1: Getting access to the data

In [1]:
import polars as pl

In [2]:
#get all the files

common_players = pl.scan_csv("./csv/common_player_info.csv")
game_info = pl.scan_csv("./csv/game_info.csv")
game = pl.scan_csv("./csv/game.csv")
inactive_players = pl.scan_csv("./csv/inactive_players.csv")
line_score = pl.scan_csv("./csv/line_score.csv")
other_stats = pl.scan_csv("./csv/other_stats.csv")
play_by_play = pl.scan_csv("./csv/play_by_play.csv")
player = pl.scan_csv("./csv/player.csv")
team = pl.scan_csv("./csv/team.csv")

In [3]:
#create dictionary for the files and print out the columns of each file
files = {
    "common_players": common_players,
    "game_info": game_info,
    "game": game,
    "inactive_players": inactive_players,
    "line_score": line_score,
    "other_stats": other_stats,
    "play_by_play": play_by_play,
    "player": player,
    "team": team
}

for name, col in files.items():
    print(name.upper() + ":")
    print(col.columns)

COMMON_PLAYERS:
['person_id', 'first_name', 'last_name', 'display_first_last', 'display_last_comma_first', 'display_fi_last', 'player_slug', 'birthdate', 'school', 'country', 'last_affiliation', 'height', 'weight', 'season_exp', 'jersey', 'position', 'rosterstatus', 'games_played_current_season_flag', 'team_id', 'team_name', 'team_abbreviation', 'team_code', 'team_city', 'playercode', 'from_year', 'to_year', 'dleague_flag', 'nba_flag', 'games_played_flag', 'draft_year', 'draft_round', 'draft_number', 'greatest_75_flag']
GAME_INFO:
['game_id', 'game_date', 'attendance', 'game_time']
GAME:
['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home', 'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home', '

/var/folders/fc/v3bqm1y90l17f7kqdx225_r80000gn/T/ipykernel_13124/3149967515.py:16: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(col.columns)


In [4]:
for name, path in [
    ("common_players", "./csv/common_player_info.csv"),
    ("game_info", "./csv/game_info.csv"),
    ("game", "./csv/game.csv"),
    ("inactive_players", "./csv/inactive_players.csv"),
    ("line_score", "./csv/line_score.csv"),
    ("other_stats", "./csv/other_stats.csv"),
    ("play_by_play", "./csv/play_by_play.csv"),
    ("player", "./csv/player.csv"),
    ("team", "./csv/team.csv")
]:
    df = pl.read_csv(path, ignore_errors=True)
    print(f"{name.upper()}: {df.shape}")

COMMON_PLAYERS: (4171, 33)
GAME_INFO: (58053, 4)
GAME: (65698, 55)
INACTIVE_PLAYERS: (110191, 9)
LINE_SCORE: (58053, 43)
OTHER_STATS: (28271, 26)
PLAY_BY_PLAY: (13592899, 34)
PLAYER: (4831, 5)
TEAM: (30, 7)


Step 2: Identifying starters for every team in every game

In [5]:
all_starters = play_by_play.filter(pl.col("period") == 1) #first quarter
all_starters = all_starters.group_by(["game_id", "player1_team_id"]).agg(
    pl.col("player1_id").value_counts(sort=True)
    .struct.field("player1_id")
    .head(5)
    .alias("lineup")
) #in order of event

#match to game results
game_results = game.select(["game_id", "team_id_home", "team_id_away", "wl_home"])
training_data = game_results.join(all_starters.rename({"player1_team_id": "team_id_home", "lineup": "home_lineup"}), on="game_id") #get home lineup
training_data = training_data.join(all_starters.rename({"player1_team_id": "team_id_away", "lineup": "away_lineup"}), on="game_id") #get away lineup

ready_data = training_data.collect()


In [6]:
#create stats report

game_dates = game.select([
    pl.col("game_id"),
    pl.col("game_date").str.slice(0,4).alias("season")
])
pbp_with_season = play_by_play.join(game_dates, on="game_id")
#using the codes to assign variable values
made_shot = pl.col("eventmsgtype") == 1
missed_shot = pl.col("eventmsgtype") == 2
free_throw = pl.col("eventmsgtype") == 3
rebound = pl.col("eventmsgtype") == 4
turnover = pl.col("eventmsgtype") == 5


#check to see if descriptions have info 
description_has_3pts = (pl.col("homedescription").str.contains("3PT") | pl.col("visitordescription").str.contains("3PT"))
description_has_assist = (pl.col("homedescription").str.contains("AST") | pl.col("visitordescription").str.contains("AST"))
description_has_steal = (pl.col("homedescription").str.contains("STEAL") | pl.col("visitordescription").str.contains("STEAL"))
description_has_rebound = (pl.col("homedescription").str.contains("REBOUND") | pl.col("visitordescription").str.contains("REBOUND"))
description_has_block = (pl.col("homedescription").str.contains("BLOCK") | pl.col("visitordescription").str.contains("BLOCK"))

#stats report
pts = (
        pl.when(free_throw).then(1)
        .when(made_shot & description_has_3pts).then(3)
        .when(made_shot).then(2)
        .otherwise(0)
    )
stats_report_by_player = pbp_with_season.with_columns(
    points = pts,
    rebounds = pl.when(rebound).then(1).otherwise(0),
    steal = (turnover & description_has_steal),
    assist = (made_shot & description_has_assist),
    block = (missed_shot & description_has_block),
)


#There are three players to attribute the stats to: p1 who makes the points and rebounds, p2 with the assits and steals, and p3 with the blocks

p1_stats = stats_report_by_player.group_by("player1_id", "season").agg(
    pl.col("points").sum().alias("total_points"), #getting points
    pl.col("rebounds").sum().alias("total_rebounds") #getting rebounds
).rename({"player1_id": "player_id"}) #removing the number from the name

p2_stats = stats_report_by_player.group_by("player2_id", "season").agg(
    pl.col("assist").sum().alias("total_assists"), #getting assists
    pl.col("steal").sum().alias("total_steals") #getting steals
).rename({"player2_id": "player_id"}) #removing the number from the name

p3_stats = stats_report_by_player.group_by("player3_id", "season").agg(
    pl.col("block").sum().alias("total_blocks"), #getting blocks
).rename({"player3_id": "player_id"}) #removing the number from the name

#overall report
report = p1_stats.join(p2_stats, on=["player_id", "season"], how="outer", coalesce=True)
report = report.join(p3_stats, on=["player_id", "season"], how="outer", coalesce=True).fill_null(0)

#calculating averages per game

games_played = pbp_with_season.group_by(["player1_id", "season"]).agg(pl.col("game_id").n_unique().alias("gp")).rename({"player1_id": "player_id"}) #gets total games played

final_report = report.join(games_played, on=["player_id", "season"], how="inner", coalesce=True).select([
    pl.col("player_id"),
    pl.col("season"),
    (pl.col("total_points") / pl.col("gp")).alias("avg_pts"), #gets percentage pts of total pts
    (pl.col("total_rebounds") / pl.col("gp")).alias("avg_rbds"), #gets percentage rebounds of total rebounds
    (pl.col("total_assists") / pl.col("gp")).alias("avg_assists"), #gets percentage assists of total assists
    (pl.col("total_blocks") / pl.col("gp")).alias("avg_blocks"), #gets percentage blocks of total blocks
    (pl.col("total_steals") / pl.col("gp")).alias("avg_steals") #gets percentage steals of total steals
]).fill_null(0)





/var/folders/fc/v3bqm1y90l17f7kqdx225_r80000gn/T/ipykernel_13124/3365399054.py:56: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  report = p1_stats.join(p2_stats, on=["player_id", "season"], how="outer", coalesce=True)
/var/folders/fc/v3bqm1y90l17f7kqdx225_r80000gn/T/ipykernel_13124/3365399054.py:57: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  report = report.join(p3_stats, on=["player_id", "season"], how="outer", coalesce=True).fill_null(0)


In [7]:
game_dates = game.select([pl.col("game_id"), pl.col("game_date").str.slice(0,4).alias("season")]) #gets the year date of the game and calls it season
roster = play_by_play.select([pl.col("player1_id").alias("player_id"), pl.col("player1_team_id").alias("team_id"), pl.col("game_id")]) #gets the players and teams and renames the variables
roster = roster.join(game_dates, on="game_id") #joins
roster = roster.select(["player_id", "team_id", "season"]) #selects those columns
roster = roster.unique() #gets unique players

#player names
player_names = player.select([pl.col("id").alias("player_id"), pl.col("full_name").alias("player_name")]) #renames variables and selects them
final_report_df = final_report.join(player_names, on="player_id", how="left") #joins
final_report_df = final_report_df.join(roster, on=["player_id", "season"], how="inner") #joins
final_report_df = final_report_df.collect()

Getting the best starting lineup from a team and a single stat line from said team

In [8]:
def best_starting_lineup(df, team_id, year):
    df_new = df.filter((pl.col("team_id") == team_id) & (pl.col("season").str.contains(str(year)))) #filter for the team wanted and year wanted
    #add all the metrics into a score
    df_new = df_new.unique(subset=["player_id"])
    df_new = df_new.with_columns(score = (pl.col("avg_pts") + pl.col("avg_assists") + pl.col("avg_rbds") + pl.col("avg_blocks") + pl.col("avg_steals")))
    df_new = df_new.sort("score", descending=True) #sort by descending order
    df_new = df_new.head(5) #get the top 5

    results = df_new
    starting_five_ids = results.select("player_id").to_series().to_list() #get list of players
    starting_five_names = results.select("player_name").to_series().to_list()
    return starting_five_ids, starting_five_names

def single_stat_line(starting_lineup_dataframe): #sums each person's avgs into one stat line and returns an array
    pts = starting_lineup_dataframe["avg_pts"].sum() 
    assists = starting_lineup_dataframe["avg_assists"].sum()
    rebounds = starting_lineup_dataframe["avg_rbds"].sum()
    blocks = starting_lineup_dataframe["avg_blocks"].sum()
    steals = starting_lineup_dataframe["avg_steals"].sum()
    ret = np.array([pts, assists, rebounds, blocks, steals])
    return ret


Training the model

In [9]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

stats_lookup = report.join(games_played, on=["player_id", "season"], how="inner").select([
    pl.col("player_id").cast(pl.Int64),
    pl.col("season"),
    (pl.col("total_points") / pl.col("gp")).alias("avg_pts"), #gets percentage pts of total pts
    (pl.col("total_rebounds") / pl.col("gp")).alias("avg_rbds"), #gets percentage rebounds of total rebounds
    (pl.col("total_assists") / pl.col("gp")).alias("avg_assists"), #gets percentage assists of total assists
    (pl.col("total_blocks") / pl.col("gp")).alias("avg_blocks"), #gets percentage blocks of total blocks
    (pl.col("total_steals") / pl.col("gp")).alias("avg_steals")
]).collect()

stats_dict = { #allows for a quicker lookup with a dictionary
    (row["player_id"], row["season"]): [
        row["avg_pts"], row["avg_rbds"], row["avg_assists"], row["avg_blocks"], row["avg_steals"], 
    ]
    for row in stats_lookup.to_dicts()  
}

if "season" not in ready_data.columns: #get the year and name that as the season
    game_seasons = game.select([pl.col("game_id"), pl.col("game_date").str.slice(0,4).alias("season")]).collect()
    ready_data = ready_data.join(game_seasons, on="game_id")

#create fit vectors
gaps = []
wins_losses = []

for row in ready_data.iter_rows(named=True):
    season = row["season"]
    home_ids = row["home_lineup"]
    away_ids = row["away_lineup"]
    
    try:
        home_stats = np.array([stats_dict.get((p_id, season), [0, 0, 0, 0, 0]) for p_id in home_ids]).sum(axis=0) #get the stats for the home lineup
        away_stats = np.array([stats_dict.get((p_id, season), [0, 0, 0, 0, 0]) for p_id in away_ids]).sum(axis=0) #get the stats for the away lineup
        gap = np.concatenate([home_stats, away_stats, [1]]) #calculate the gap between the home and away
        gaps.append(gap)

        if row["wl_home"] == "W": #if home won the game
            wins_losses.append(1)
        else:
            wins_losses.append(0)
    except Exception as e:
        continue

scaler = StandardScaler()
gaps_scaled = scaler.fit_transform(gaps)
gaps_train, gaps_test, wl_train, wl_test = train_test_split(gaps_scaled, wins_losses, test_size = 0.2, random_state = 42)
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42).fit(gaps_scaled, wins_losses)


Simulating the 7 game series

In [10]:
def run_simulation(first_team_id, first_team_year, second_team_id, second_team_year):
    team_name_1 = team.filter(pl.col("id") == first_team_id).select("full_name").collect().to_series()[0] #gets the first team's name attached to the id
    team_name_2 = team.filter(pl.col("id") == second_team_id).select("full_name").collect().to_series()[0] #gets the second team's name attached to the id
    
    ids_a, names_a = best_starting_lineup(final_report_df, first_team_id, first_team_year) #gets the ids and names of the first team
    ids_b, names_b = best_starting_lineup(final_report_df, second_team_id, second_team_year) #gets the ids and names of the second team

    print("-------------------------------------------------------------------------------------------------")
    print(f"Matchup: {first_team_year} {team_name_1} vs. {second_team_year} {team_name_2}")
    print(f"{team_name_1} Starters: {", ".join(names_a)}")
    print(f"{team_name_2} Starters: {", ".join(names_b)}")
    print("-------------------------------------------------------------------------------------------------")

    #gets team stats
    def get_team_stats(ids, team_id, year):
        
        res = final_report_df.filter(
            (pl.col("player_id").is_in(ids)) & 
            (pl.col("team_id") == team_id) & 
            (pl.col("season").str.contains(str(year)))
            ).unique(subset=["player_id"]).select([ #get the sums of the 5 metrics
            pl.col("avg_pts").sum(),
            pl.col("avg_rbds").sum(),
            pl.col("avg_assists").sum(),
            pl.col("avg_blocks").sum(),
            pl.col("avg_steals").sum(),
        ])
        if hasattr(res, "collect"): #if it's lazy frame
            res = res.collect()
        return res.to_numpy()[0]

    stats_a = get_team_stats(ids_a, first_team_id, first_team_year)  #stats of the first team
    stats_b = get_team_stats(ids_b, second_team_id, second_team_year) #stats of the second team

    print("Stats A:", stats_a)
    print("Stats B:", stats_b)

    gap = np.concatenate([stats_a, stats_b, [0]]).reshape(1, -1) #base probability of which team is better per stat taking in home court advantage

    gap_scaled = scaler.transform(gap)
    win_probability_team = model.predict_proba(gap_scaled)[0][1]

    if win_probability_team > 0.5: #checks to see which team has a higher than random chance of winning
        certainty = win_probability_team
        favored = "Team 1"
    else:
        certainty = 1 - win_probability_team
        favored = "Team 2"

    print(f"{favored} is {certainty * 100:.2f}% favorite when playing on a nuetral court")
    print("-------------------------------------------------------------------------------------------------")

    #7 game series
    a_wins = 0
    b_wins = 0

    for game in range(1,8): #7 game series
        home_flag = 1 if game in [1,2,5,7] else 0
        game_gap = np.concatenate([stats_a, stats_b, [home_flag]]).reshape(1, -1)
        

        game_gap_scaled = scaler.transform(game_gap)
        current_probability = model.predict_proba(game_gap_scaled.reshape(1, -1))[0][1] #predict using home court advantage

        if np.random.random() < current_probability: #check to see if the current probability of winning is greater than random chance
            a_wins += 1
            print(f"Game {game}: Team 1 wins")
        else:
            b_wins += 1
            print(f"Game {game}: Team 2 wins")

        if a_wins == 4: #won series
            winner = "Team 1"
            break
        if b_wins == 4:
            winner = "Team 2"
            break

    print(f"Final Result: {winner} wins the series {max(a_wins, b_wins)} - {min(a_wins, b_wins)}!")
    print("-------------------------------------------------------------------------------------------------")


Should return the starting lineup names, and who would win the series in how many games

In [11]:
run_simulation(1610612744, 2016, 1610612760, 1998)

-------------------------------------------------------------------------------------------------
Matchup: 2016 Golden State Warriors vs. 1998 Oklahoma City Thunder
Golden State Warriors Starters: Kevin Durant, Stephen Curry, Draymond Green, Klay Thompson, Harrison Barnes
Oklahoma City Thunder Starters: Gary Payton, Vin Baker, Detlef Schrempf, Hersey Hawkins, Dale Ellis
-------------------------------------------------------------------------------------------------
Stats A: [107.0306439   31.45030781  21.55303159   3.31268809   6.63507847]
Stats B: [78.40934405 26.19435212 17.3253742   1.7709223   6.11143151]
Team 1 is 63.94% favorite when playing on a nuetral court
-------------------------------------------------------------------------------------------------
Game 1: Team 2 wins
Game 2: Team 1 wins
Game 3: Team 2 wins
Game 4: Team 2 wins
Game 5: Team 2 wins
Final Result: Team 2 wins the series 4 - 1!
----------------------------------------------------------------------------------

In [12]:
import pandas as pd
pd.set_option('display.max_rows', None)
print(team.select(["id", "full_name"]).collect().to_pandas())

            id               full_name
0   1610612737           Atlanta Hawks
1   1610612738          Boston Celtics
2   1610612739     Cleveland Cavaliers
3   1610612740    New Orleans Pelicans
4   1610612741           Chicago Bulls
5   1610612742        Dallas Mavericks
6   1610612743          Denver Nuggets
7   1610612744   Golden State Warriors
8   1610612745         Houston Rockets
9   1610612746    Los Angeles Clippers
10  1610612747      Los Angeles Lakers
11  1610612748              Miami Heat
12  1610612749         Milwaukee Bucks
13  1610612750  Minnesota Timberwolves
14  1610612751           Brooklyn Nets
15  1610612752         New York Knicks
16  1610612753           Orlando Magic
17  1610612754          Indiana Pacers
18  1610612755      Philadelphia 76ers
19  1610612756            Phoenix Suns
20  1610612757  Portland Trail Blazers
21  1610612758        Sacramento Kings
22  1610612759       San Antonio Spurs
23  1610612760   Oklahoma City Thunder
24  1610612761         To

In [13]:
print("Model accuracy:", model.score(gaps_test, wl_test))

Model accuracy: 0.5976113405514009
